# Previsão de Taxa de Metabolismo Basal (BMR) para App de Acompanhamento Nutricional

## 1. O Problema
O sedentarismo e a má alimentação são problemas crescentes de saúde pública. Para combater esse cenário, nosso projeto foca no monitormaento de saúde digital através de um aplicativo de **acompanhamento nutricional**.

Um dos grandes desafios em dietas e reeducação alimentar é personalizar a recomendação calórica diária para cada usuário. Fórmulas genéricas podem não capturar nuances específicas. Precisamos de um sistema que calcule com precisão a **Taxa de Metabolismo Basal (BMR)** (quantidade de calorias que o corpo gasta em repouso), servindo de base para o plano alimentar e promovendo bem-estar, prevenção e qualidade de vida.

## 2. O que a IA faz
No contexto do aplicativo, o modelo atuará como o **motor de personalização metabólica**. 

Em vez de depender apenas de equações estáticas pré-definidas, a IA utiliza dados históricos de pacientes para aprender a relação intrínseca entre as características físicas (idade, peso, altura e gênero) e o BMR real. Quando um novo usuário se cadastra no app, a IA recebe suas informações físicas e **prevê automaticamente o seu BMR**, ajustando instantaneamente as metas calóricas no dashboard do aplicativo.

## 3. Dados
Os dados utilizados foram extraídos de um repositório público (Fonte: [kaggle.com/sampathrajj/bmr-dataset](https://www.kaggle.com/sampathrajj/bmr-dataset)).

O dataset `BMR_Dataset.csv` contém as seguintes informações sobre os indivíduos:
- **user_id**: Identificador único do usuário.
- **age**: Idade do usuário (anos).
- **weight**: Peso do usuário (kg).
- **height**: Altura do usuário (cm).
- **gender**: Gênero biológico ('Male' / 'Female').
- **BMR**: Taxa de Metabolismo Basal (variável alvo/target, em calorias).

Abaixo, realizamos a importação e a Análise Exploratória de Dados (EDA).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Carregando o dataset
df = pd.read_csv('BMR_Dataset.csv')
print("Dataset carregado com sucesso!")

In [ ]:
# Verificando o formato (shape) dos dados (linhas, colunas)
print(f"Formato do dataset: {df.shape[0]} linhas e {df.shape[1]} colunas\n")

# Exibindo as primeiras 5 linhas do dataframe
display(df.head())

In [ ]:
# Exibindo estatísticas descritivas (média, desvio padrão, mínimos e máximos)
display(df.describe())

# Verificando a existência de valores nulos
print("\nValores nulos por coluna:")
print(df.isnull().sum())

## 4. Pré-processamento
Nesta etapa, preparamos os dados para serem inseridos no modelo.
1. **Tratamento de Nulos**: Removeremos as linhas com valores ausentes (NaN), pois a Regressão Linear do scikit-learn não lida nativamente com dados em branco.
2. **Remoção de colunas irrelevantes**: Removeremos o `user_id`, pois é apenas um identificador e não tem poder preditivo.
3. **Encoding**: O modelo compreende apenas números, mas a coluna `gender` contém texto ('Male' e 'Female'). Vamos convertê-la usando codificação (`LabelEncoder`).
4. **Separação de Features e Target**: Dividiremos as variáveis independentes ($X$) e a variável dependente ($y$, que é o nosso alvo BMR).
5. **Divisão em Treino e Teste**: Separaremos os dados em duas partes: uma para treinar o modelo (aprendizado) e outra para testar seu poder de generalização.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# 1. Tratamento de valores nulos (NaN)
df_processed = df.dropna()

# 2. Removendo a coluna de ID
df_processed = df_processed.drop(columns=['user_id'])

# 3. Encoding da variável 'gender'
le = LabelEncoder()
df_processed['gender'] = le.fit_transform(df_processed['gender'])

# Mostrando o mapeamento efetuado
print("Mapeamento de Gênero:", dict(zip(le.classes_, le.transform(le.classes_))))

# 4. Separando as features (X) da variável alvo (y)
X = df_processed.drop(columns=['BMR'])
y = df_processed['BMR']

# 5. Dividindo em conjuntos de Treino (80%) e Teste (20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"\nTamanho do conjunto de Treino: {X_train.shape[0]} amostras")
print(f"Tamanho do conjunto de Teste: {X_test.shape[0]} amostras")

## 5. Modelo
Para a previsão de BMR, escolhemos o algoritmo de **Regressão Linear Múltipla** utilizando a biblioteca `scikit-learn`.

### Justificativa da Escolha:
- **Natureza do Problema**: O cálculo de BMR na literatura médica é intrinsecamente um problema linear (uma combinação de adição e subtração de variáveis corporais). Uma Regressão Linear captura essas relações perfeitamente.
- **Interpretabilidade (Caixa Branca)**: No desenvolvimento para saúde, é crucial entender como a IA toma decisões. A regressão linear nos fornece coeficientes transparentes sobre o peso que ela dá a cada variável corporal.
- **Eficiência Computacional**: O modelo resultante é extremamente leve, permitindo processamento rápido e baixo custo operacional de nuvem quando exposto pela API do aplicativo móvel.

In [ ]:
from sklearn.linear_model import LinearRegression

# Instanciando a estrutura do modelo matemático
modelo_bmr = LinearRegression()

print("Modelo 'LinearRegression' instanciado com sucesso!")

## 6. Treinamento
Nesta fase, a Inteligência Artificial "ajusta" sua reta multidimensional utilizando o conjunto de treino. Ela calcula os coeficientes perfeitos para cada variável a fim de minimizar o erro entre sua previsão e o valor real documentado no dataset.

In [ ]:
# Treinando o modelo com os dados históricos
modelo_bmr.fit(X_train, y_train)

print("==== Parâmetros Aprendidos pela IA ====")
print(f"Intercepto (Valor Constante Base): {modelo_bmr.intercept_:.2f}\n")

print("Coeficientes/Pesos das Features:")
for feature, coef in zip(X.columns, modelo_bmr.coef_):
    print(f" - {feature}: {coef:.2f}")

# Estes coeficientes indicam, por exemplo, o quanto o BMR sobe ou desce a cada 1 ano de idade ou 1kg de peso.

## 7. Resultados e Eficiência
Vamos avaliar como o modelo se comporta diante de **dados nunca vistos** (nosso conjunto de teste). Para isso, utilizaremos as seguintes métricas:

- **MAE (Erro Médio Absoluto)**: Mostra em média por quantas calorias a IA está errando (para mais ou para menos).
- **RMSE (Raiz do Erro Quadrático Médio)**: Uma métrica que penaliza erros grandes, ajudando a identificar se existem grandes distorções.
- **R² (Coeficiente de Determinação)**: Representa em percentual (%) a quantidade de variância do BMR que nossa IA consegue explicar.

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Prevendo o BMR com os dados de teste
y_pred = modelo_bmr.predict(X_test)

# Avaliando as métricas estatísticas
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("=== Métricas de Desempenho ===")
print(f"MAE  (Erro Absoluto Médio)  : {mae:.2f} kcal")
print(f"RMSE (Erro Quadrático Médio) : {rmse:.2f} kcal")
print(f"R²   (Poder de Explicação)   : {r2:.4f}")
print("==============================\n")

# Plotando gráfico real x previsto para melhor visualização
plt.figure(figsize=(8, 5))
plt.scatter(y_test, y_pred, alpha=0.6, color='blue')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], '--r', linewidth=2)
plt.title('BMR Real vs BMR Previsto pela IA')
plt.xlabel('BMR Real (kcal)')
plt.ylabel('BMR Previsto (kcal)')
plt.grid(True)
plt.show()

print("Interpretação:")
print(f"O modelo explica extremanamente bem {(r2*100):.1f}% das variações no gasto calórico dos usuários.")
print(f"A margem de erro está na média de {mae:.0f} calorias, atestando alta precisão.")

## 8. Conclusão
A experimentação comprova a viabilidade técnica de utilizar Inteligência Artificial para previsões metabólicas neste projeto.

**Aplicação Real no Projeto:**
1. Usuários fornecerão apenas `idade`, `peso`, `altura` e `gênero`. O modelo (hospedado numa nuvem/API) executará este pipeline instantaneamente.
2. Caso o usuário registre novo peso no app futuramente, o serviço rodará uma nova inferência imediatamente, readequando o plano dietético da semana.
3. Ao prover dietas balizadas pelas verdadeiras necessidades fisiológicas de repouso (BMR assertivo), minimizamos as falhas na dieta, engajamos o usuário com resultados consistentes e cumprimos o objetivo principal de expansão em serviços de **saúde digital, prevenção e bem-estar**.